In [10]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import os
print(os.listdir())
import category_encoders as ce
import optuna
from sklearn.metrics import mean_pinball_loss

import warnings
warnings.filterwarnings('ignore')

['house_price_competition_day_2_failed_trying.ipynb', 'house_price_advanced_view.ipynb', 'addr_kmeans.pkl', 'submission.csv', 'house_price_advanced_models.ipynb', 'my_model_submission.csv1', 'submission5.csv', 'my_model_submission4.csv', 'house_price', 'addr_umap.pkl', 'Day1.ipynb', 'titanic', 'house-prices-advanced-regression-techniques.zip', 'titanic.zip', 'my_model_submission3.csv', 'house_price_competition_view_day2.ipynb', 'Untitled.ipynb', 'my_model_submission0.csv', 'X_umap.npy', 'addr_tfidf.pkl', 'Day2 Housing Price.ipynb', 'pca_model.pkl', 'my_model_submission.csv', 'predictions.csv', 'umap_model.pkl', 'predictions4.csv', 'my_model_submission1.csv', 'competition_day7.ipynb', 'house_price_view.ipynb', 'home-data-for-ml-course.zip', 'Untitled2.ipynb', '.ipynb_checkpoints', 'home-data-for-ml-course', 'house_price.ipynb', 'predictions3.csv', 'failed_trying_with_nns_day_9.ipynb', 'not_so_bad_trying_with_lightgbm_day_8.ipynb', 'my_model_submission2.csv', 'best_interval_model.pth', '

In [12]:
train = pd.read_csv('house_price/dataset.csv')
test = pd.read_csv('house_price/test.csv')
print ("Data is loaded!")

quantitative = [f for f in train.columns if train.dtypes[f] != 'object']
quantitative.remove('sale_price')
quantitative.remove('id')
qualitative = [f for f in train.columns if train.dtypes[f] == 'object']

sns.set_style("whitegrid")
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing)

sns.set_style("whitegrid")
missing = test.isnull().sum()
missing = missing[missing > 0]
print(missing)

def dataset_fill_null(obj):
    obj['subdivision'].fillna('Unknown', inplace=True)
    obj.drop(columns=['sale_nbr'], inplace=True)
    obj['submarket'].fillna('Unknown', inplace=True)


dataset_fill_null(train)
dataset_fill_null(test)
print(train.shape)
print(test.shape)

# 构造原始地址字段
train_ID = train['id']
test_ID = test['id']
# Now drop the  'Id' colum since it's unnecessary for  the prediction process.
drop_cols=['id',#row_id,没有任何信息.
           'golf',#20万数据 198756都是0,基本没什么信息了.
           'view_rainier',#20万数据,198588都是0.
           'view_skyline',#20万数据,198517都是0.
           'view_lakesamm',#20万数据,198776都是0.
           'view_otherwater',#20万数据,198473都是0.
           'view_other',#20万数据,198833都是0.
          ]
train.drop(drop_cols, axis=1, inplace=True)
test_raw = test.drop(drop_cols, axis=1, inplace=True)

# Deleting outliers
train.reset_index(drop=True, inplace=True)
# We use the numpy fuction log1p which  applies log(1+x) to all elements of the column
# train["sale_price"] = np.log1p(train["sale_price"])
y = train.sale_price.reset_index(drop=True)

def preprocess_and_encode(df, y=None, encoder_bundle=None, drop_high_card=True):
    df = df.copy()
    
    # ================= 清洗阶段 ================= #
    if encoder_bundle is None:
        # 训练阶段
        address_text = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
        vectorizer = CountVectorizer(max_features=1000, token_pattern=r'\b\w+\b', ngram_range=(1, 2))
        address_vec = vectorizer.fit_transform(address_text)
        svd = TruncatedSVD(n_components=50, random_state=42)
        address_pca = svd.fit_transform(address_vec)
    else:
        # 推理阶段
        vectorizer = encoder_bundle['vectorizer']
        svd = encoder_bundle['svd']
        address_text = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
        address_vec = vectorizer.transform(address_text)
        address_pca = svd.transform(address_vec)

    address_df = pd.DataFrame(address_pca, columns=[f'address_pca_{i+1}' for i in range(address_pca.shape[1])], index=df.index)
    df = df.drop(columns=['city', 'subdivision'], errors='ignore')
    df = pd.concat([df, address_df], axis=1)

    # 补充缺失
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    df[cat_cols] = df[cat_cols].fillna('None')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 日期和派生
    if 'sale_date' in df.columns:
        df['sale_year'] = pd.to_datetime(df['sale_date']).dt.year
        df['sale_month'] = pd.to_datetime(df['sale_date']).dt.month
        df = df.drop(columns='sale_date')
        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)

    # 编码分类变量
    cat_cols = df.select_dtypes(include='object').columns
    low_card_cols = []
    high_card_cols = []
    
    for col in cat_cols:
        try:
            n_unique = df[col].nunique()
            if isinstance(n_unique, (int, np.integer)):
                if n_unique <= 50:
                    low_card_cols.append(col)
                else:
                    high_card_cols.append(col)
            else:
                print(f"[跳过] {col} 的 nunique 结果不是标量: {n_unique}")
        except Exception as e:
            print(f"[异常] {col}: {e}")

    X_cat = pd.get_dummies(df[low_card_cols], dummy_na=True)

    if encoder_bundle is None:
        target_encoder = ce.TargetEncoder()
        X_target = target_encoder.fit_transform(df[high_card_cols], y) if y is not None else pd.DataFrame(index=df.index)
    else:
        target_encoder = encoder_bundle['target_encoder']
        X_target = target_encoder.transform(df[high_card_cols]) if high_card_cols else pd.DataFrame(index=df.index)
    X_target.columns = [f"{col}_te" for col in high_card_cols]

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    if encoder_bundle is None:
        scaler = StandardScaler()
        X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)
    else:
        scaler = encoder_bundle['scaler']
        X_num_scaled = pd.DataFrame(scaler.transform(df[num_cols]), columns=num_cols, index=df.index)

    X_final_df = pd.concat([X_num_scaled, X_cat, X_target], axis=1)

    if encoder_bundle is None:
        encoder_bundle = {
            'vectorizer': vectorizer,
            'svd': svd,
            'target_encoder': target_encoder,
            'scaler': scaler
        }

    return X_final_df, encoder_bundle



def add_knn_price_features(df, base_df=None, lat_col='latitude', lon_col='longitude',
                           target_col='target', ks=[5, 10, 20]):
    if base_df is None:
        base_df = df  # 默认自己为近邻池

    coords_query = df[[lat_col, lon_col]].values
    coords_base = base_df[[lat_col, lon_col]].values
    tree = KDTree(coords_base, metric='euclidean')

    base_targets = base_df[target_col].values
    knn_features = {}

    for k in ks:
        dists, indices = tree.query(coords_query, k=k)
        neighbor_targets = base_targets[indices]

        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    for col, val in knn_features.items():
        df[col] = val

    return df


def add_radius_knn_features(df, base_df, lat_col='latitude', lon_col='longitude', target_col='target', radius=0.01, max_neighbors=100):
    coords = np.radians(base_df[[lat_col, lon_col]])
    tree = BallTree(coords, metric='haversine')  # 地理距离计算更准

    df_coords = np.radians(df[[lat_col, lon_col]])
    indices = tree.query_radius(df_coords, r=radius)

    # 每个样本找到若干邻居索引后，聚合
    agg_means, agg_stds, agg_counts = [], [], []
    base_targets = base_df[target_col].values

    for idxs in indices:
        if len(idxs) > 1:
            if len(idxs) > max_neighbors:
                idxs = idxs[:max_neighbors]
            neigh_vals = base_targets[idxs]
            agg_means.append(np.mean(neigh_vals))
            agg_stds.append(np.std(neigh_vals))
            agg_counts.append(len(idxs))
        else:
            agg_means.append(np.nan)
            agg_stds.append(np.nan)
            agg_counts.append(0)

    df['radius_knn_mean'] = agg_means
    df['radius_knn_std'] = agg_stds
    df['radius_knn_count'] = agg_counts

    return df


def add_knn_price_features_radius(df, base_df, lat_col='latitude', lon_col='longitude',
                                   target_col='sale_price', radius=0.01, max_neighbors=100):
    df = df.copy()
    
    coords_query = df[[lat_col, lon_col]].astype(np.float32).values
    coords_base = base_df[[lat_col, lon_col]].astype(np.float32).values
    targets_base = base_df[target_col].astype(np.float32).values

    tree = KDTree(coords_base, leaf_size=40, metric='euclidean')
    neighbor_indices = tree.query_radius(coords_query, r=radius)

    means, stds, ranges = [], [], []

    for i, inds in enumerate(neighbor_indices):
        # 排除自身（仅当 base_df 是 df）
        if base_df is df:
            inds = inds[inds != i]
        
        if len(inds) == 0:
            means.append(np.nan)
            stds.append(np.nan)
            ranges.append(np.nan)
        else:
            if len(inds) > max_neighbors:
                inds = inds[:max_neighbors]
            vals = targets_base[inds]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
            ranges.append(np.max(vals) - np.min(vals))

    df[f'knn_radius_mean'] = means
    df[f'knn_radius_std'] = stds
    df[f'knn_radius_range'] = ranges

    return df

def preprocess_and_encode(df, y=None, encoder_bundle=None, drop_high_card=True):
    df = df.copy()
    
    # ================= 清洗阶段 ================= #
    if encoder_bundle is None:
        # 训练阶段
        address_text = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
        vectorizer = CountVectorizer(max_features=1000, token_pattern=r'\b\w+\b', ngram_range=(1, 2))
        address_vec = vectorizer.fit_transform(address_text)
        svd = TruncatedSVD(n_components=50, random_state=42)
        address_pca = svd.fit_transform(address_vec)
    else:
        # 推理阶段
        vectorizer = encoder_bundle['vectorizer']
        svd = encoder_bundle['svd']
        address_text = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
        address_vec = vectorizer.transform(address_text)
        address_pca = svd.transform(address_vec)

    address_df = pd.DataFrame(address_pca, columns=[f'address_pca_{i+1}' for i in range(address_pca.shape[1])], index=df.index)
    df = df.drop(columns=['city', 'subdivision'], errors='ignore')
    df = pd.concat([df, address_df], axis=1)

    # 补充缺失
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    df[cat_cols] = df[cat_cols].fillna('None')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 日期和派生
    if 'sale_date' in df.columns:
        df['sale_year'] = pd.to_datetime(df['sale_date']).dt.year
        df['sale_month'] = pd.to_datetime(df['sale_date']).dt.month
        df = df.drop(columns='sale_date')
        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)

    # 编码分类变量
    cat_cols = df.select_dtypes(include='object').columns
    low_card_cols = []
    high_card_cols = []
    
    for col in cat_cols:
        try:
            n_unique = df[col].nunique()
            if isinstance(n_unique, (int, np.integer)):
                if n_unique <= 50:
                    low_card_cols.append(col)
                else:
                    high_card_cols.append(col)
            else:
                print(f"[跳过] {col} 的 nunique 结果不是标量: {n_unique}")
        except Exception as e:
            print(f"[异常] {col}: {e}")

    X_cat = pd.get_dummies(df[low_card_cols], dummy_na=True)

    if encoder_bundle is None:
        target_encoder = ce.TargetEncoder()
        X_target = target_encoder.fit_transform(df[high_card_cols], y) if y is not None else pd.DataFrame(index=df.index)
    else:
        target_encoder = encoder_bundle['target_encoder']
        X_target = target_encoder.transform(df[high_card_cols]) if high_card_cols else pd.DataFrame(index=df.index)
    X_target.columns = [f"{col}_te" for col in high_card_cols]

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    if encoder_bundle is None:
        scaler = StandardScaler()
        X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)
    else:
        scaler = encoder_bundle['scaler']
        X_num_scaled = pd.DataFrame(scaler.transform(df[num_cols]), columns=num_cols, index=df.index)

    X_final_df = pd.concat([X_num_scaled, X_cat, X_target], axis=1)

    if encoder_bundle is None:
        encoder_bundle = {
            'vectorizer': vectorizer,
            'svd': svd,
            'target_encoder': target_encoder,
            'scaler': scaler
        }

    return X_final_df, encoder_bundle



def add_knn_price_features(df, base_df=None, lat_col='latitude', lon_col='longitude',
                           target_col='target', ks=[5, 10, 20]):
    if base_df is None:
        base_df = df  # 默认自己为近邻池

    coords_query = df[[lat_col, lon_col]].values
    coords_base = base_df[[lat_col, lon_col]].values
    tree = KDTree(coords_base, metric='euclidean')

    base_targets = base_df[target_col].values
    knn_features = {}

    for k in ks:
        dists, indices = tree.query(coords_query, k=k)
        neighbor_targets = base_targets[indices]

        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    for col, val in knn_features.items():
        df[col] = val

    return df


def add_radius_knn_features(df, base_df, lat_col='latitude', lon_col='longitude', target_col='target', radius=0.01, max_neighbors=100):
    coords = np.radians(base_df[[lat_col, lon_col]])
    tree = BallTree(coords, metric='haversine')  # 地理距离计算更准

    df_coords = np.radians(df[[lat_col, lon_col]])
    indices = tree.query_radius(df_coords, r=radius)

    # 每个样本找到若干邻居索引后，聚合
    agg_means, agg_stds, agg_counts = [], [], []
    base_targets = base_df[target_col].values

    for idxs in indices:
        if len(idxs) > 1:
            if len(idxs) > max_neighbors:
                idxs = idxs[:max_neighbors]
            neigh_vals = base_targets[idxs]
            agg_means.append(np.mean(neigh_vals))
            agg_stds.append(np.std(neigh_vals))
            agg_counts.append(len(idxs))
        else:
            agg_means.append(np.nan)
            agg_stds.append(np.nan)
            agg_counts.append(0)

    df['radius_knn_mean'] = agg_means
    df['radius_knn_std'] = agg_stds
    df['radius_knn_count'] = agg_counts

    return df


def add_knn_price_features_radius(df, base_df, lat_col='latitude', lon_col='longitude',
                                   target_col='sale_price', radius=0.01, max_neighbors=100):
    df = df.copy()
    
    coords_query = df[[lat_col, lon_col]].astype(np.float32).values
    coords_base = base_df[[lat_col, lon_col]].astype(np.float32).values
    targets_base = base_df[target_col].astype(np.float32).values

    tree = KDTree(coords_base, leaf_size=40, metric='euclidean')
    neighbor_indices = tree.query_radius(coords_query, r=radius)

    means, stds, ranges = [], [], []

    for i, inds in enumerate(neighbor_indices):
        # 排除自身（仅当 base_df 是 df）
        if base_df is df:
            inds = inds[inds != i]
        
        if len(inds) == 0:
            means.append(np.nan)
            stds.append(np.nan)
            ranges.append(np.nan)
        else:
            if len(inds) > max_neighbors:
                inds = inds[:max_neighbors]
            vals = targets_base[inds]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
            ranges.append(np.max(vals) - np.min(vals))

    df[f'knn_radius_mean'] = means
    df[f'knn_radius_std'] = stds
    df[f'knn_radius_range'] = ranges

    return df

# 应用预处理
# 对训练集（自己做自己）

X_train_raw, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42
)

test_raw = test.copy()  # 或者正确读取原始测试集

X_train = add_knn_price_features(X_train_raw, base_df=X_train_raw, target_col='sale_price')
X_val = add_knn_price_features(X_val, base_df=X_train_raw, target_col='sale_price')
X_train_full = add_knn_price_features(train, base_df=train, target_col='sale_price')
# 对测试集（用训练集做 base）
test = add_knn_price_features(test_raw, base_df=X_train_raw, target_col='sale_price')

X_train = add_knn_price_features_radius(X_train, base_df=X_train_raw, target_col='sale_price')
X_val = add_knn_price_features_radius(X_val, base_df=X_train_raw, target_col='sale_price')
X_train_full = add_knn_price_features_radius(X_train_full, base_df=train, target_col='sale_price')
# 对测试集（用训练集做 base）
test = add_knn_price_features_radius(test, base_df=X_train_raw, target_col='sale_price')

X_train = X_train.drop(['sale_price'], axis=1)
X_val = X_val.drop(['sale_price'], axis=1)
X_train_full = X_train_full.drop(['sale_price'], axis=1)
X_train, encoder_bundle = preprocess_and_encode(X_train, y_train)
X_val, _ = preprocess_and_encode(X_val, encoder_bundle=encoder_bundle)
test, _ = preprocess_and_encode(test, encoder_bundle=encoder_bundle)
X_train_full, _ = preprocess_and_encode(X_train_full, y)

def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):

    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)

# ========= 数据划分 =========
X_tr, X_val_, y_tr, y_val_ = X_train, X_val, y_train, y_val

def train_quantile_model(X, y, alpha, params):
    params = params.copy()
    params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'verbosity': -1
    })
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y,
              eval_set=[(X_val_, y_val_)],
               callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
                )
    return model

def objective(trial):
    scores = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(X_train_full):
        X_tr = X_train_full.iloc[train_idx]
        y_tr = y.iloc[train_idx]
        X_va = X_train_full.iloc[val_idx]
        y_va = y.iloc[val_idx]
        
        common_params = {
            'n_estimators': trial.suggest_int('n_estimators', 500, 3000, step=500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 5, 64),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100, step=10),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
            'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 5.0),
            'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 5.0),
        }

        model_low = train_quantile_model(X_tr, y_tr, alpha=0.05, params=common_params)
        model_high = train_quantile_model(X_tr, y_tr, alpha=0.95, params=common_params)
        y_low = model_low.predict(X_va)
        y_high = model_high.predict(X_va)

        score, coverage = winkler_score(y_va, y_low, y_high, alpha=0.1, return_coverage=True)
        if coverage < 0.9:
            score += 1e5
            
        scores.append(score)

    return np.mean(scores)

# ========= 启动搜索 =========
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, n_jobs=4)
print("Best params:", study.best_params)

# ===================== Gamma压缩优化模块 ===================== #
from scipy.optimize import minimize

print("\n🔍 开始优化压缩比例 gamma0, gamma1 ...")

def winkler_with_gamma(gammas, y_true, lower, upper, alpha=0.1):
    gamma0, gamma1 = gammas
    mid = (upper + lower) / 2
    width = (upper - lower)
    lower_adj = mid - gamma0 * width / 2
    upper_adj = mid + gamma1 * width / 2
    return winkler_score(y_true, lower_adj, upper_adj, alpha)

# 使用最佳参数训练的模型再预测一次（防止值过期）
params = study.best_params
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in kf.split(X_train_full):
    X_tr = X_train_full.iloc[train_idx]
    y_tr = y.iloc[train_idx]
    X_va = X_train_full.iloc[val_idx]
    y_va = y.iloc[val_idx]

    model_low = train_quantile_model(X_tr, y_tr, alpha=0.05, params=params)
    model_high = train_quantile_model(X_tr, y_tr, alpha=0.95, params=params)
    pred_low = model_low.predict(X_va)
    pred_high = model_high.predict(X_va)

# 初始 gamma = 1 表示不缩放
initial_gamma = [1.0, 1.0]
result = minimize(
    winkler_with_gamma,
    x0=initial_gamma,
    args=(y_val_, pred_lower, pred_upper),
    bounds=[(0.5, 1.5), (0.5, 1.5)],
    method='Nelder-Mead'
)

best_gamma0, best_gamma1 = result.x
print(f"✅ 最佳 gamma0: {best_gamma0:.3f}, gamma1: {best_gamma1:.3f}")

# 应用压缩
mid = (pred_upper + pred_lower) / 2
width = (pred_upper - pred_lower)
final_lower = mid - best_gamma0 * width / 2
final_upper = mid + best_gamma1 * width / 2

final_score, final_coverage = winkler_score(y_val_, final_lower, final_upper, alpha=0.1, return_coverage=True)
print(f"🎯 Optimized Winkler Score: {final_score:.2f}")
print(f"📈 Coverage after compression: {final_coverage:.4f}")


Data is loaded!
sale_nbr       42182
subdivision    17550
submarket       1717
dtype: int64
sale_nbr       42412
subdivision    17550
submarket       1718
dtype: int64
(200000, 46)
(200000, 45)


[I 2025-07-13 17:53:44,707] A new study created in memory with name: no-name-540609d5-368e-422f-bb36-2fd10b1d3308


Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 9078.37
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[981]	valid_0's quantile: 8915.91
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 33669
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's quantile: 8636.55
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1590]	valid_0's quantile: 33167.7
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2385]	valid_0's quantile: 8988.8

[I 2025-07-13 17:59:21,128] Trial 1 finished with value: 420048.0661118786 and parameters: {'n_estimators': 500, 'learning_rate': 0.05353452265957205, 'num_leaves': 40, 'min_data_in_leaf': 60, 'feature_fraction': 0.6170423212088504, 'bagging_fraction': 0.9514612153366552, 'bagging_freq': 2, 'lambda_l1': 0.7791284848089136, 'lambda_l2': 1.0456661289846159}. Best is trial 1 with value: 420048.0661118786.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 8241.81
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's quantile: 10467.9
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's quantile: 36475.4
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 32142.4
Did not meet early stopping. Best iteration is:
[2999]	valid_0's quantile: 8344.3
Did not meet early stopping. Best iteration is:
[2985]	valid_0's quantile: 35905.8
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's quantile: 1

...

Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 8323.15
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's quantile: 32982.3
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's quantile: 32525.2


[I 2025-07-14 01:49:35,548] Trial 98 finished with value: 409890.15289281996 and parameters: {'n_estimators': 2000, 'learning_rate': 0.027312705452070745, 'num_leaves': 29, 'min_data_in_leaf': 90, 'feature_fraction': 0.6894980770549316, 'bagging_fraction': 0.9018574921959619, 'bagging_freq': 3, 'lambda_l1': 4.42983397700693, 'lambda_l2': 0.6285192248822591}. Best is trial 96 with value: 408645.93117666553.


Did not meet early stopping. Best iteration is:
[1999]	valid_0's quantile: 8340.97
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1885]	valid_0's quantile: 32710.2


[I 2025-07-14 01:50:20,759] Trial 99 finished with value: 409502.38082794106 and parameters: {'n_estimators': 2000, 'learning_rate': 0.027392860901296105, 'num_leaves': 28, 'min_data_in_leaf': 90, 'feature_fraction': 0.6869680452572676, 'bagging_fraction': 0.9402412278491988, 'bagging_freq': 3, 'lambda_l1': 4.467292803689839, 'lambda_l2': 0.1302412008775815}. Best is trial 96 with value: 408645.93117666553.


Best params: {'n_estimators': 2000, 'learning_rate': 0.03319123198352938, 'num_leaves': 33, 'min_data_in_leaf': 90, 'feature_fraction': 0.622465133612465, 'bagging_fraction': 0.9483007457095504, 'bagging_freq': 3, 'lambda_l1': 4.671764586834628, 'lambda_l2': 0.19608416829744352}

🔍 开始优化压缩比例 gamma0, gamma1 ...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1981]	valid_0's quantile: 8604.94
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's quantile: 33147.9
✅ 最佳 gamma0: 1.181, gamma1: 1.500
🎯 Optimized Winkler Score: 798628.44
📈 Coverage after compression: 0.8299


In [14]:
s, c = winkler_score(y_val, pred_lower, pred_upper, alpha=0.1, return_coverage=True)

In [15]:
print(s,c)

835055.8864514176 0.741125


In [16]:
# 用最佳参数再跑一次交叉验证评分
def evaluate_best_params_cv(params):
    scores = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(X_train_full):
        X_tr = X_train_full.iloc[train_idx]
        y_tr = y.iloc[train_idx]
        X_va = X_train_full.iloc[val_idx]
        y_va = y.iloc[val_idx]

        model_low = train_quantile_model(X_tr, y_tr, alpha=0.05, params=params)
        model_high = train_quantile_model(X_tr, y_tr, alpha=0.95, params=params)
        pred_low = model_low.predict(X_va)
        pred_high = model_high.predict(X_va)

        score, coverage = winkler_score(y_va, pred_low, pred_high, alpha=0.1, return_coverage=True)
        scores.append(score)
        print(f"Fold Winkler: {score:.0f} | Coverage: {coverage:.3f}")

    print(f"\n✅ 5-Fold CV Winkler Avg: {np.mean(scores):,.0f}")
    return np.mean(scores)

evaluate_best_params_cv(study.best_params)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1751]	valid_0's quantile: 8857.11
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1882]	valid_0's quantile: 32841.2
Fold Winkler: 313577 | Coverage: 0.857
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's quantile: 8293.92
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1725]	valid_0's quantile: 32301.3
Fold Winkler: 309073 | Coverage: 0.853
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 8121.29
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1995]	valid_0's quantile: 32136.6
Fold Winkler: 307332 | Coverage: 0.855
Training until validation scores don't improve for 50 roun

308645.93117666553

In [17]:
print("\n🔍 开始优化压缩比例 gamma0, gamma1 ...")

def winkler_with_gamma(gammas, y_true, lower, upper, alpha=0.1):
    gamma0, gamma1 = gammas
    mid = (upper + lower) / 2
    width = (upper - lower)
    lower_adj = mid - gamma0 * width / 2
    upper_adj = mid + gamma1 * width / 2
    return winkler_score(y_true, lower_adj, upper_adj, alpha)

# 使用最佳参数训练的模型再预测一次（防止值过期）
params = study.best_params
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in kf.split(X_train_full):
    X_tr = X_train_full.iloc[train_idx]
    y_tr = y.iloc[train_idx]
    X_va = X_train_full.iloc[val_idx]
    y_va = y.iloc[val_idx]

    model_low = train_quantile_model(X_tr, y_tr, alpha=0.05, params=params)
    model_high = train_quantile_model(X_tr, y_tr, alpha=0.95, params=params)
    pred_low = model_low.predict(X_va)
    pred_high = model_high.predict(X_va)

# 初始 gamma = 1 表示不缩放
initial_gamma = [1.0, 1.0]
result = minimize(
    winkler_with_gamma,
    x0=initial_gamma,
    args=(y_val, pred_lower, pred_upper),
    bounds=[(0.5, 1.5), (0.5, 1.5)],
    method='Nelder-Mead'
)

best_gamma0, best_gamma1 = result.x
print(f"✅ 最佳 gamma0: {best_gamma0:.3f}, gamma1: {best_gamma1:.3f}")

# 应用压缩
mid = (pred_upper + pred_lower) / 2
width = (pred_upper - pred_lower)
final_lower = mid - best_gamma0 * width / 2
final_upper = mid + best_gamma1 * width / 2

final_score, final_coverage = winkler_score(y_val, final_lower, final_upper, alpha=0.1, return_coverage=True)
print(f"🎯 Optimized Winkler Score: {final_score:.2f}")
print(f"📈 Coverage after compression: {final_coverage:.4f}")


🔍 开始优化压缩比例 gamma0, gamma1 ...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1751]	valid_0's quantile: 8857.11
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1882]	valid_0's quantile: 32841.2
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's quantile: 8293.92
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1725]	valid_0's quantile: 32301.3
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 8121.29
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1995]	valid_0's quantile: 32136.6
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 8181.45


In [ ]:
print("\n🔍 开始优化压缩比例 gamma0, gamma1 ...")

def winkler_with_gamma(gammas, y_true, lower, upper, alpha=0.1):
    gamma0, gamma1 = gammas
    mid = (upper + lower) / 2
    width = (upper - lower)
    lower_adj = mid - gamma0 * width / 2
    upper_adj = mid + gamma1 * width / 2
    return winkler_score(y_true, lower_adj, upper_adj, alpha)

# 使用最佳参数训练的模型再预测一次（防止值过期）
params = study.best_params
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in kf.split(X_train_full):
    X_tr = X_train_full.iloc[train_idx]
    y_tr = y.iloc[train_idx]
    X_va = X_train_full.iloc[val_idx]
    y_va = y.iloc[val_idx]

    model_low = train_quantile_model(X_tr, y_tr, alpha=0.05, params=params)
    model_high = train_quantile_model(X_tr, y_tr, alpha=0.95, params=params)
    pred_low = model_low.predict(X_va)
    pred_high = model_high.predict(X_va)

# 初始 gamma = 1 表示不缩放
initial_gamma = [1.0, 1.0]
result = minimize(
    winkler_with_gamma,
    x0=initial_gamma,
    args=(y_val, pred_lower, pred_upper),
    bounds=[(0.5, 1.5), (0.5, 1.5)],
    method='Nelder-Mead'
)

best_gamma0, best_gamma1 = result.x
print(f"✅ 最佳 gamma0: {best_gamma0:.3f}, gamma1: {best_gamma1:.3f}")

# 应用压缩
mid = (pred_upper + pred_lower) / 2
width = (pred_upper - pred_lower)
final_lower = mid - best_gamma0 * width / 2
final_upper = mid + best_gamma1 * width / 2

final_score, final_coverage = winkler_score(y_val, final_lower, final_upper, alpha=0.1, return_coverage=True)
print(f"🎯 Optimized Winkler Score: {final_score:.2f}")
print(f"📈 Coverage after compression: {final_coverage:.4f}")

In [19]:
final_model_low = train_quantile_model(X_train_full, y, alpha=0.05, params=best_params)
final_model_high = train_quantile_model(X_train_full, y, alpha=0.95, params=best_params)


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's quantile: 8113.48
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's quantile: 31767.4


In [20]:
prew_low = final_model_low.predict(test)

In [21]:
prew_high = final_model_high.predict(test)

In [22]:
submission = pd.DataFrame({
    'id': test_ID,
    'pi_lower': prew_low,
    'pi_upper': prew_high
})
submission.to_csv('submission6.csv', index=False)

In [24]:
pred_low = model_low.predict(test)
pred_high = model_high.predict(test)

In [25]:
submission = pd.DataFrame({
    'id': test_ID,
    'pi_lower': prew_low,
    'pi_upper': prew_high
})
submission.to_csv('submission6.csv', index=False)